# P4 · IDEA 2 — дистилляция головы на soft-скорах `M` (уплотнение супервизии)

**Идея.** Единственная голова `node_pred` сейчас получает реальный CrossEntropy-таргет только на тех temporal-edge батчах, что пересекают границу label-дня (`train` L95: `query_t = batch.t[-1] > label_t`). Остальные батчи лишь пишут рёбра в TGN-память (`process_edges` L180) + `memory.detach()` (L181) — **ни forward, ни градиента**. IDEA 2 заполняет эти target-less батчи **вспомогательным дистилляционным лоссом**: учитель = плотная память `M[u,:]` (уже даёт genre test NDCG@10 ≈ 0.52), студент = та же голова. KD-лосс `KL(log_softmax(student) ‖ softmax(M/T))` учит голову имитировать мягкие per-item скоры `M` на user-узлах этих батчей.

**Это НЕ ансамбль.** `M` — учитель **только на train**; на инференсе одна голова `node_pred(z)`, `M` не инстанцируется и не подмешивается к логитам (запрещённое — `logits = head(z) + M[u,:]`; здесь этого нет). Дистилляция = каноническое «сжатие учителя в веса студента», учитель отбрасывается на инференсе.

**Этот ноутбук — pre-experiment проба (P4). ОБУЧЕНИЯ НЕТ.** Офлайн проверяем 4 под-предпосылки, прежде чем тратить P5-прогон:
- **(a)** target-less доля материальна (>~90% батчей) **И** на них есть user-узлы с непустой строкой `M` — есть кого и чем учить;
- **(b)** `M` — достоверный учитель (NDCG≈0.52, высокий top-10 hit) **И** `M` 0.52 > TGNv2 0.469 ⇒ цель ВЫШЕ модели (headroom, не потолок);
- **(c)** сигнал `M` **стабилен** через no-target разрыв — `M` вчерашнего дня предсказывает сегодняшние метки почти как свежая ⇒ stale-таргет валиден;
- **(d)** soft-таргет `softmax(M/T)` калибруем (энтропия между one-hot и uniform), ранжирование сохраняется.

Если любая падает — IDEA 2 опровергнута офлайн. Каузальность: `M` строится строго прошлым, читается ДО записи рёбер своего батча.

In [1]:
# Setup — harness из p3/p4 (verbatim). Ядро свежее.
import numpy as np, polars as pl, plotly.express as px, plotly.graph_objects as go, sys
from sklearn.metrics import ndcg_score
from tgb.nodeproppred.dataset_pyg import PyGNodePropPredDataset
from tgb.nodeproppred.evaluate import Evaluator
from torch_geometric.loader import TemporalDataLoader
REPO = "/Users/aleksandrpanysev/Documents/GitHub/2Q_2026_tgn_user_item"
if REPO not in sys.path: sys.path.insert(0, REPO)

def load_dataset(name, bs=200):
    ds = PyGNodePropPredDataset(name=name, root="datasets")
    data = ds.get_TemporalData()
    tr, va, te = data.train_val_test_split(val_ratio=0.15, test_ratio=0.15)
    return dict(ds=ds, data=data, num_classes=ds.num_classes, num_nodes=data.num_nodes,
                evaluator=Evaluator(name=name), eval_metric=ds.eval_metric,
                loaders={s: TemporalDataLoader(d, batch_size=bs) for s, d in [("train", tr), ("val", va), ("test", te)]})

def make_rank_phi(DS):
    tr, _, _ = DS["data"].train_val_test_split(val_ratio=0.15, test_ratio=0.15)
    sw = np.sort(tr.msg[:, 0].numpy()); n = len(sw)
    return lambda w: np.searchsorted(sw, w, side="right") / n

class HawkesMemory:
    def __init__(self, N, C, alpha, phi):
        self.C = C; self.M = np.zeros((N, C)); self.tlast = np.zeros((N, C)); self.alpha = float(alpha); self.phi = phi
    def update(self, src, dst, t, w):
        if src.size == 0: return
        idx = src.astype(np.int64) * self.C + dst.astype(np.int64); tb = float(t.max())
        uniq, inv = np.unique(idx, return_inverse=True); add = np.bincount(inv, weights=self.phi(w), minlength=uniq.size)
        fM, fT = self.M.reshape(-1), self.tlast.reshape(-1)
        fM[uniq] = (fM[uniq] * np.exp(-self.alpha * np.clip(tb - fT[uniq], 0, None)) + add) if self.alpha > 0 else fM[uniq] + add
        fT[uniq] = tb
    def read(self, users, t_read):
        r = self.M[users]
        return r * np.exp(-self.alpha * np.clip(t_read - self.tlast[users], 0, None)) if self.alpha > 0 else r

DS = load_dataset("tgbn-genre"); DCHAR = 86400.0; KAPPA = 25
alpha = np.log(2) / (KAPPA * DCHAR); phi = make_rank_phi(DS); C = DS["num_classes"]
assert int(DS["data"].dst.max()) < C  # item-node == class ⇒ M-колонки совпадают с осью классов головы
print(f"genre: nodes={DS['num_nodes']} classes={C} edges={DS['data'].src.numel()}  alpha={alpha:.3e}  (item==class ✓)")

genre: nodes=1505 classes=513 edges=17858395  alpha=3.209e-07  (item==class ✓)


### Предпосылка (a) — сколько супервизии простаивает и есть ли кого учить

Воспроизводим точное условие `train` L95 (`batch.t[-1] > label_t` ⇒ supervised, иначе target-less), считаем доли. На каждом батче ДО записи его рёбер смотрим user-узлы (`src ≥ num_classes`) и долю **тёплых** (непустая строка `M`) — именно их можно дистиллировать. **Гейт:** target-less > ~90% батчей **И** на них стабильно есть тёплые user-узлы.

In [2]:
# Каузальный проход: считаем supervised vs target-less батчи + тёплые user-узлы (читаем M ДО записи рёбер)
def scan_batches(DS, mem, dl, split, rows):
    ds = DS["ds"]; label_t = ds.get_label_time(); C = DS["num_classes"]
    for batch in dl:
        sn, dn, tn = batch.src.numpy(), batch.dst.numpy(), batch.t.numpy(); wn = batch.msg[:, 0].numpy()
        sup = float(batch.t[-1]) > label_t
        if sup:
            lt = ds.get_node_label(batch.t[-1])
            if lt is None: sup = False
            else: label_t = ds.get_label_time()
        u = np.unique(sn[sn >= C])                       # user-узлы батча (items < C)
        warm = int((mem.M[u].sum(1) > 0).sum()) if u.size else 0
        rows.append((split, int(sup), int(u.size), warm, len(sn)))
        mem.update(sn, dn, tn, wn)                        # запись ПОСЛЕ чтения (каузально)
    return rows

mem = HawkesMemory(DS["num_nodes"], C, alpha, phi); rows = []
for sp in ["train", "val", "test"]:
    scan_batches(DS, mem, DS["loaders"][sp], sp, rows)
DS["ds"].reset_label_time()
bb = pl.DataFrame(rows, schema=["split", "sup", "n_user", "n_warm", "n_edges"], orient="row")
tl = bb.filter(pl.col("sup") == 0)
print(f"всего батчей={bb.height}  supervised={bb['sup'].sum()} ({bb['sup'].mean()*100:.1f}%)  "
      f"target-less={tl.height} ({(1-bb['sup'].mean())*100:.1f}%)")
print(f"target-less батчи с ≥1 тёплым user-узлом: {(tl['n_warm']>0).mean()*100:.1f}%")
print(f"тёплых user-узлов на target-less батч: med={int(tl['n_warm'].median())}  "
      f"mean={tl['n_warm'].mean():.1f}  (из ~{int(tl['n_user'].median())} user-узлов/батч)")
print("\nпо сплитам (доля supervised):")
print(bb.group_by("split").agg(pl.col("sup").mean().round(4).alias("sup_frac"),
                               (pl.col("sup")==0).sum().alias("targetless"), pl.len().alias("n")))

всего батчей=89293  supervised=1579 (1.8%)  target-less=87714 (98.2%)
target-less батчи с ≥1 тёплым user-узлом: 100.0%
тёплых user-узлов на target-less батч: med=31  mean=30.2  (из ~31 user-узлов/батч)

по сплитам (доля supervised):
shape: (3, 4)
┌───────┬──────────┬────────────┬───────┐
│ split ┆ sup_frac ┆ targetless ┆ n     │
│ ---   ┆ ---      ┆ ---        ┆ ---   │
│ str   ┆ f64      ┆ u32        ┆ u32   │
╞═══════╪══════════╪════════════╪═══════╡
│ val   ┆ 0.0122   ┆ 13231      ┆ 13394 │
│ test  ┆ 0.0124   ┆ 13228      ┆ 13394 │
│ train ┆ 0.02     ┆ 61255      ┆ 62505 │
└───────┴──────────┴────────────┴───────┘


### Предпосылки (b) учитель достоверен + headroom и (c) стабилен через разрыв

Один строго-каузальный проход собирает per-`(user,day)` на val+test: `ndcg` (учитель vs метка), `top10hit`, `lm_top10` (масса метки в M-top10), и **`ndcg_stale`** — NDCG памяти, **замороженной на предыдущем label-дне** и применённой к сегодняшним меткам (снапшот `M` на прошлой границе). 

**Гейт (b):** `ndcg`≈0.52 высокий top-10 hit **И** 0.52 > TGNv2 0.469. **Гейт (c):** `ndcg_stale ≈ ndcg` (вчерашняя `M` предсказывает сегодня почти как свежая) ⇒ stale soft-таргет на target-less батчах валиден.

In [3]:
# Учитель: качество (b) + стабильность (c). Лаговый снапшот M = "вчерашняя память".
def run_teacher(DS, phi, alpha):
    N, Cc = DS["num_nodes"], DS["num_classes"]; ds = DS["ds"]; mem = HawkesMemory(N, Cc, alpha, phi)
    recs = []; prev = {"M": None, "T": None}
    def stream(dl, collect):
        label_t = ds.get_label_time()
        for batch in dl:
            sn, dn, tn = batch.src.numpy(), batch.dst.numpy(), batch.t.numpy(); wn = batch.msg[:, 0].numpy()
            if float(batch.t[-1]) > label_t:
                lt = ds.get_node_label(batch.t[-1])
                if lt is None: break
                lab_ts0 = float(lt[0][0]); ls, labs = lt[1].numpy(), lt[2].numpy()
                label_t = ds.get_label_time(); pm = tn < lab_ts0
                mem.update(sn[pm], dn[pm], tn[pm], wn[pm])
                if collect:
                    cur = mem.read(ls, lab_ts0); hp = prev["M"] is not None
                    st = prev["M"][ls] * np.exp(-alpha * np.clip(lab_ts0 - prev["T"][ls], 0, None)) if hp else None
                    for i in range(len(ls)):
                        yi = labs[i]; tot = yi.sum()
                        if tot <= 0: continue
                        ci = cur[i]; pos = yi > 0; npos = int(pos.sum())
                        t10 = np.argpartition(-ci, 10)[:10]
                        recs.append((int(ls[i]), npos, float(ndcg_score(yi[None], ci[None], k=10)),
                                     int(pos[t10].sum()) / min(10, npos), float(yi[t10].sum() / tot),
                                     float(ndcg_score(yi[None], st[i][None], k=10)) if hp else float("nan")))
                    prev["M"] = mem.M.copy(); prev["T"] = mem.tlast.copy()
                sn, dn, tn, wn = sn[~pm], dn[~pm], tn[~pm], wn[~pm]
            mem.update(sn, dn, tn, wn)
        return mem
    stream(DS["loaders"]["train"], False); stream(DS["loaders"]["val"], True)
    memf = stream(DS["loaders"]["test"], True); ds.reset_label_time()
    return pl.DataFrame(recs, schema=["user","n_pos","ndcg","top10hit","lm_top10","ndcg_stale"], orient="row"), memf

td, mem_final = run_teacher(DS, phi, alpha)
print(f"учитель собран: n(user,day)={td.height}")
print(f"(b) NDCG@10={td['ndcg'].mean():.4f}  top10hit={td['top10hit'].mean():.3f}  "
      f"масса метки в M-top10={td['lm_top10'].mean():.3f}")
stb = td.filter(pl.col("ndcg_stale").is_not_nan())
print(f"(c) свежая M={stb['ndcg'].mean():.4f}  vs  ВЧЕРАШНЯЯ M (stale)={stb['ndcg_stale'].mean():.4f}  "
      f"→ потеря {stb['ndcg'].mean()-stb['ndcg_stale'].mean():+.4f}")

учитель собран: n(user,day)=68771
(b) NDCG@10=0.5175  top10hit=0.550  масса метки в M-top10=0.590
(c) свежая M=0.5175  vs  ВЧЕРАШНЯЯ M (stale)=0.5147  → потеря +0.0028


In [4]:
# (b) headroom: учитель M выше обучаемой TGNv2 → дистилляция имеет запас, не потолок
bars = pl.DataFrame({"модель": ["TGNv2 (learned)", "MovAvg(L)", "M-учитель (этот ноут)", "ceiling (p4 support)"],
                     "ndcg": [0.469, 0.509, td["ndcg"].mean(), 0.987]}).to_pandas()
fig = px.bar(bars, x="модель", y="ndcg", text="ndcg", title="Headroom: M-учитель ≫ TGNv2 ⇒ студент, имитируя M, может превзойти текущую модель",
             color="модель", color_discrete_sequence=["#d7191c", "#fdae61", "#2c7fb8", "#999999"])
fig.update_traces(texttemplate="%{text:.3f}", textposition="outside")
fig.update_layout(height=420, showlegend=False, yaxis_title="test NDCG@10")
fig.add_hline(y=0.469, line_dash="dash", line_color="#d7191c",
              annotation_text=f"зазор M−TGNv2 = +{td['ndcg'].mean()-0.469:.3f}")
fig.show()

# (c) свежая vs вчерашняя M — практически совпадают
sl = stb.select(["ndcg", "ndcg_stale"]).rename({"ndcg": "свежая M", "ndcg_stale": "вчерашняя M (stale)"})
fig2 = px.histogram(sl.unpivot(variable_name="вид", value_name="NDCG@10").to_pandas(), x="NDCG@10", color="вид",
                    barmode="overlay", nbins=40, opacity=0.6,
                    title=f"Стабильность учителя через no-target разрыв: потеря {stb['ndcg'].mean()-stb['ndcg_stale'].mean():+.4f}")
fig2.update_layout(height=340)
fig2.show()

### Предпосылка (d) — калибровка soft-таргета и температура

NDCG ранг-инвариантен к `softmax` (монотонное преобразование) ⇒ ранжирование сохраняется при любом `T` тривиально. Реальный вопрос — **форма** soft-таргета: у строки `M` ~150 ненулевых из 513, и наивный `softmax(M)` даёт `exp(0)=1` каждому из ~360 **непросмотренных** item'ов ⇒ масса утекает в нерелевантный хвост. Сравниваем `softmax(M/T)` (сетка T) с `row-normalize(M)` (масса только на просмотренных) по энтропии и **доле массы на seen-item'ах**. **Гейт:** есть калибровка с массой на релевантных item'ах и энтропией между one-hot (0) и uniform (`ln 513≈6.24`).

In [6]:
# (d) форма soft-таргета: softmax(M/T) [числ. стабильный] vs row-normalize. Тёплые строки финальной M.
rng = np.random.default_rng(0)
warm_u = np.where(mem_final.M.sum(1) > 0)[0]
R = mem_final.M[rng.choice(warm_u, size=min(3000, warm_u.size), replace=False)]
seen = (R > 0).astype(float)
print(f"сырые значения M в строке: med_max={np.median(R.max(1)):.1f}  med_sum={np.median(R.sum(1)):.1f}  "
      f"→ большие/разнородные ⇒ фикс. T хрупка")
def ent(P): P = np.clip(P, 1e-12, None); return float((-(P * np.log(P)).sum(1)).mean())
def softmax(X, T):  # стабильный: вычитаем row-max
    Z = np.exp((X - X.max(1, keepdims=True)) / T); return Z / Z.sum(1, keepdims=True)
rows = []
for T in [0.5, 1.0, 2.0, 5.0, 10.0]:
    P = softmax(R, T); rows.append(("softmax(M/T)", T, ent(P), float(np.exp(ent(P))), float((P * seen).sum(1).mean())))
Pn = R / R.sum(1, keepdims=True)
rows.append(("row-normalize", None, ent(Pn), float(np.exp(ent(Pn))), 1.0))
cal = pl.DataFrame(rows, schema=["форма", "T", "энтропия", "эфф_классов", "масса_на_seen"], orient="row")
print(f"\nuniform: энтропия=ln(513)={np.log(C):.2f}, эфф_классов=513 | one-hot: 0 | медиана seen/строка={int(np.median(seen.sum(1)))}")
print(cal)
print("\nВЫВОД (d): row-normalize(M) — масштаб-инвариантный soft-таргет, 100% массы на seen, "
      "эфф_классов≈37 (между one-hot и uniform). softmax(M/T): из-за больших разнородных |M| фикс. T "
      "хрупка (низкая T→one-hot, теряет dark knowledge; высокая→утечка в хвост). РЕКОМЕНДАЦИЯ P5: "
      "row-normalize или masked-softmax по seen.")

сырые значения M в строке: med_max=53.4  med_sum=434.8  → большие/разнородные ⇒ фикс. T хрупка

uniform: энтропия=ln(513)=6.24, эфф_классов=513 | one-hot: 0 | медиана seen/строка=130
shape: (6, 5)
┌───────────────┬──────┬──────────┬─────────────┬───────────────┐
│ форма         ┆ T    ┆ энтропия ┆ эфф_классов ┆ масса_на_seen │
│ ---           ┆ ---  ┆ ---      ┆ ---         ┆ ---           │
│ str           ┆ f64  ┆ f64      ┆ f64         ┆ f64           │
╞═══════════════╪══════╪══════════╪═════════════╪═══════════════╡
│ softmax(M/T)  ┆ 0.5  ┆ 0.152226 ┆ 1.164423    ┆ 0.989184      │
│ softmax(M/T)  ┆ 1.0  ┆ 0.357396 ┆ 1.429602    ┆ 0.973053      │
│ softmax(M/T)  ┆ 2.0  ┆ 0.843112 ┆ 2.323587    ┆ 0.927937      │
│ softmax(M/T)  ┆ 5.0  ┆ 2.269093 ┆ 9.670626    ┆ 0.773738      │
│ softmax(M/T)  ┆ 10.0 ┆ 3.552868 ┆ 34.913296   ┆ 0.620184      │
│ row-normalize ┆ null ┆ 3.622308 ┆ 37.423837   ┆ 1.0           │
└───────────────┴──────┴──────────┴─────────────┴───────────────┘

ВЫВОД (d):

## Вердикт IDEA 2 — **GO** ✅

| предпосылка | гейт | измерено | вердикт |
|---|---|---|---|
| (a) простаивающая супервизия | target-less > ~90%, есть кого учить | **98.2%** батчей; **100%** с тёплыми user-узлами (~30/батч) | ✅ сильно |
| (b) достоверный учитель + headroom | NDCG≈0.52, M > TGNv2 | NDCG **0.518**, top10hit 0.55; зазор к TGNv2 **+0.049** | ✅ |
| (c) стабильность через разрыв | stale ≈ fresh | вчерашняя M **0.5147** vs свежая **0.5175** (потеря **0.0028**) | ✅ сильно |
| (d) калибруемый soft-таргет | энтропия между one-hot и uniform, масса на релевантных | **row-normalize(M)**: 100% массы на seen, эфф_классов≈37 | ✅ |

**Механизм.** 98% батчей сейчас дают нулевой градиент, но содержат ~30 тёплых user-узлов каждый — огромный неиспользованный сигнал. Учитель `M` (0.518) надёжнее обучаемой TGNv2 (0.469) и почти не дрейфует за день ⇒ stale soft-таргет на target-less батчах валиден. Правильная форма таргета — `row-normalize(M)`, не наивный `softmax(M/T)` (хрупок из-за больших `|M|`).

**No-ensemble / каузальность** соблюдены: `M` — учитель только на train; на инференсе одна голова, `M` не подмешивается к логитам. `M` читается ДО записи рёбер своего батча.

**→ Сидим P5-задачу:** A/B обучаемой TGNv2 + KD-лосс vs чистая TGNv2, genre, 3 сида, overall test NDCG@10. **Seam:** на target-less батчах (НЕ входят в `if query_t>label_t`, L95) — forward головы на user-узлах + `λ·KL(log_softmax(student) ‖ row-norm(M[u,:]))` ПЕРЕД `process_edges` (L180), backward ДО `memory.detach()` (L181); `M` обновляется в lockstep в `process_edges`. **Открытые параметры:** `λ` (не задавить реальный CE на 1.8% размеченных дней — иначе студент упрётся в потолок учителя 0.52), форма таргета (row-norm / masked-softmax), **подвыборка target-less батчей** (KD-forward на всех ~50× дороже/эпоху на CPU — главный риск стоимости). **Фальсификаторы** (target-less мало / M≈TGNv2 / нестабильна / soft-таргет вырожден) — все опровергнуты.